In [1]:
import requests
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

import pandas as pd
import math

#### acessando o site

In [39]:
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

In [40]:
url = "https://www.energystar.gov/partner-resources/partner-list/all-partner-list"

In [41]:
driver.get(url)

#### clicando em Submit para aparecer a tabela

In [42]:
iframe = WebDriverWait(driver,30).until(EC.presence_of_element_located((By.TAG_NAME, "iframe")))
driver.switch_to.frame(iframe)

In [43]:
bt = WebDriverWait(driver,30).until(EC.presence_of_element_located((By.ID, "submit-button")))
driver.execute_script("arguments[0].click();", bt)

#### Extraindo a tabela (várias páginas)

In [44]:
# espera até o elemento aparecer e captura suas informações
info_tabela = WebDriverWait(driver,50).until(EC.visibility_of_element_located((By.CLASS_NAME, "table-info")))

In [45]:
# encontrando o número de páginas e quantos são mostrados por página
itens_totais = int(info_tabela.get_attribute("innerText").replace("(", "").replace(")", "").split()[-1])
num_itens_pag = 20
num_paginas = math.ceil(itens_totais/num_itens_pag)

In [46]:
# encontrando a tabela
tabela = WebDriverWait(driver,10).until(EC.visibility_of_element_located((By.ID, "all-partner-list-table")))

In [47]:
# cabeçalho
cols = []
for item, col in enumerate(tabela.find_elements(By.TAG_NAME, "th")):
    cols.append(col.get_attribute("innerText"))
    cols[item] = cols[item].replace("\n", "").replace("Sort", "").replace("  ", "")

cols

['Company Name',
 'Partner Type (Programs)',
 'City',
 'State',
 'Country',
 'Awards']

In [48]:
# iterando nas páginas e capturando os dados da tabela

dados = []

time_init = time.time()
pagina = 1
while(pagina<=num_paginas):
    # encontrando a tabela
    tabela = WebDriverWait(driver,30).until(EC.visibility_of_element_located((By.ID, "all-partner-list-table")))
    
    # captura a tabela
    linhas = tabela.find_element(By.ID, "table-body").find_elements(By.TAG_NAME, "tr")

    # itera nas linhas para capturar os dados
    for linha in linhas:
        dados_linha = linha.find_elements(By.TAG_NAME, "td")
        dados_linha = [ele.text for ele in dados_linha]
        dados.append([ele for ele in dados_linha])

    # se já estiver na última página, não precisa clicar em avançar
    if pagina >= num_paginas:
        break
    
    # clicando para avançar a página
    botao_proximo = WebDriverWait(driver,30).until(EC.visibility_of_element_located((By.CLASS_NAME, "slds-button.slds-button_neutral.next-button.pagination-button")))
    driver.execute_script("arguments[0].click();", botao_proximo)

    # tempo para o site atualizar os dados da nova página
    time.sleep(1)

    pagina += 1

time_fim = time.time()
print(time_fim - time_init , ' segundos')

5025.2355234622955  segundos


In [49]:
# juntando cabeçalho + dados em um dataframe

df = pd.DataFrame(dados, columns=cols)
df

,Company Name,Partner Type (Programs),City,State,Country,Awards
0,'Ikenakea Development LLC,Home Builders and Developers; Indoor AirPlus B...,Honolulu,Hawaii,United States,
1,"""3"" Rivers, Inc.",Home Builders and Developers,Tallahassee,Florida,United States,
2,"""The Home Builder"" (Burt Theriot)",Home Builders and Developers,Houma,Louisiana,United States,
3,(ARC)form LLC Design & Construction,Home Builders and Developers; Indoor AirPlus B...,Cleveland,Ohio,United States,
4,(SCEHA) Interstate,Association and Non-Profit Organization Partne...,Columbia,South Carolina,United States,
...,...,...,...,...,...,...
48461,Zurich Alternative Asset Management,Buildings Partnership,New York,New York,United States,
48462,Zurich Homes,Home Builders and Developers,Crossville,Tennessee,United States,
48463,"Zurn Industries, Inc.",Green Lights (Historical),Dallas,Texas,United States,
48464,Zych Construction,Home Builders and Developers,Gretna,Nebraska,United States,


#### fechando a conexão

In [50]:
driver.close()